In [1]:
import sys
sys.path.append('../externals/DynaMix-python')

import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from neurodsp.spectral import compute_spectrum
from neurodsp.plts.spectral import plot_power_spectra
import os
from joblib import Parallel, delayed
from tqdm import tqdm

from src.dynamix.model.forecaster import DynaMixForecaster
from src.dynamix.utilities.plotting_eval import plot_TS_forecast, plot_3D_attractor
from src.dynamix.utilities.utilities import load_hf_model
import torch

In [2]:
# define file paths
eeg_path = '/oscar/data/sjones/shared/TDBRAIN_preprocessed/preprocessed'
metadata_path = '/oscar/home/scbao/tdbrain-model/data/TDBRAIN_participants_V2.tsv'
subj_list = os.listdir(eeg_path)
print("Number of total subjects:", len(subj_list))

# read data
metadata_df = pd.read_csv(metadata_path, delimiter='\t')

Number of total subjects: 1274


In [3]:
# generate mask to pull out MDD, DISC subjects only
subj_mask = np.isin(metadata_df['participants_ID'].values, subj_list)
discovery_mask = metadata_df['DISC/REP'].values == 'DISCOVERY'
dataset_mask = metadata_df['Dataset'].values == 'MDD-rTMS'
rTMS_mask = ~metadata_df['rTMS PROTOCOL'].isna() # excludes 1 participant

mask = np.logical_and.reduce([subj_mask, discovery_mask, dataset_mask, rTMS_mask])

df = metadata_df[mask].copy() # .copy because plan to add columns later
print("Shape after keeping MDD-rTMS, Discovery rows:", df.shape)

Shape after keeping MDD-rTMS, Discovery rows: (131, 111)


In [4]:
duplicate_ids = df["participants_ID"].value_counts()
duplicate_ids = duplicate_ids[duplicate_ids > 1].index
df = df.drop_duplicates(subset="participants_ID", keep="first")
print("Shape after keeping only first entry for each participant:", df.shape)

Shape after keeping only first entry for each participant: (123, 111)


In [5]:
# save paths to eeg numpy files for each subject in df
ec_eeg_path, eo_eeg_path, has_ses1 = list(), list(), list()

for subj_id in df['participants_ID'].values:
    subj_path = f'{eeg_path}/{subj_id}/ses-1/eeg'
    if os.path.isdir(subj_path):
        has_ses1.append(True)
        subj_files = os.listdir(subj_path)
        ec_subj_files = list(pathlib.Path(subj_path).glob('*restEC*.npy'))
        eo_subj_files = list(pathlib.Path(subj_path).glob('*restEO*.npy'))
        assert len(ec_subj_files) == len(eo_subj_files) == 1

        ec_eeg_path.append(str(ec_subj_files[0]))
        eo_eeg_path.append(str(eo_subj_files[0]))

    else:
        has_ses1.append(False)
        ec_eeg_path.append('')
        eo_eeg_path.append('')

df['ec_eeg_path'] = ec_eeg_path
df['eo_eeg_path'] = eo_eeg_path
df['has_ses1'] = has_ses1

df = df[df['has_ses1'] == True].reset_index(drop=True)
print("Shape after removing subj with no session 1 data:", df.shape)

Shape after removing subj with no session 1 data: (120, 114)


In [6]:
participant_id = "sub-87999321"

row = df[df["participants_ID"] == participant_id].iloc[0]
ec_path = row["ec_eeg_path"]
ec_eeg = np.load(ec_path, allow_pickle=True)
print(ec_eeg.keys())
eo_path = row["eo_eeg_path"]
eo_eeg = np.load(eo_path, allow_pickle=True)

np.save(f"../data/{participant_id}_ses-1_task-restEC.npy", ec_eeg)
np.save(f"../data/{participant_id}_ses-1_task-restEO.npy", eo_eeg)

dict_keys(['artifacts', 'info', 'data', 'trl', 'artidata', 'arttrl', 'Fs', 'labels', 'neighblabels'])


## Bandpower

In [7]:
# define feature extraction function
def get_bandpower_features(subj_data_path, condition, normalize=True, pool_by_region=False):
    channel_filter = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC3', 'FCz', 'FC4',
                      'T7', 'C3', 'Cz', 'C4', 'T8', 'CP3', 'CPz', 'CP4', 'P7', 'P3',
                      'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2',]
    
    regions = {
        'frontal': ['Fp1','Fp2','F7','F3','Fz','F4','F8'],
        'frontocentral': ['FC3','FCz','FC4'],
        'central': ['C3','Cz','C4','T7','T8'],
        'parietal': ['CP3','CPz','CP4','P7','P3','Pz','P4','P8'],
        'occipital': ['O1','Oz','O2']
    }

    eeg_dict = np.load(subj_data_path, allow_pickle=True)
    channel_labels = eeg_dict['labels']
    fs = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    # print(eeg_data.shape)
    assert eeg_data.shape[0] == len(channel_filter)

    bands = {
        'delta': [0.5, 4], 
        'theta': [4, 8], 
        'alpha': [8, 12], 
        'low_beta': [12, 20],
        'high_beta': [20, 30], 
        'low_gamma': [30, 40], 
        'high_gamma': [40, 80]
    }

    # construct dict with bandpower values for inputted subj
    feature_dict = {}
    channel_bandpower = {}

    for ch_idx, ch_name in enumerate(channel_filter):
        freqs, psd = compute_spectrum(eeg_data[ch_idx], fs, method='welch', avg_type='mean', nperseg=fs*2)
        
        band_powers = {}
        for band_name, (fmin, fmax) in bands.items():
            idx = (freqs >= fmin) & (freqs <= fmax)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            band_powers[band_name] = band_power

        # normalize across all band powers for each channel
        if normalize:
            total_power = sum(band_powers.values()) + 1e-8
            for band_name in band_powers:
                band_powers[band_name] /= total_power

        channel_bandpower[ch_name] = band_powers
        # only save individual channel features if not pooling
        if not pool_by_region:
            for band_name, value in band_powers.items():
                feature_dict[f'{condition}_{ch_name}_{band_name}_power'] = value

    # pool by region if requested
    if pool_by_region:
        for region_name, region_channels in regions.items():
            for band_name in bands.keys():
                values = [channel_bandpower[ch][band_name] for ch in region_channels]
                feature_dict[f'{condition}_{region_name}_{band_name}_power'] = np.mean(values)

    return feature_dict

def extract_subj_features(subj_id, ec_path, eo_path, pool_by_region=False):
    feats = {'participants_ID': subj_id}
    feats.update(get_bandpower_features(ec_path, 'EC', pool_by_region=pool_by_region))
    feats.update(get_bandpower_features(eo_path, 'EO', pool_by_region=pool_by_region))
    return feats

In [8]:
# extract features in parallel for all subjects
res = Parallel(n_jobs=16)(
    delayed(extract_subj_features)(subj_id, ec_path, eo_path)
    for subj_id, ec_path, eo_path in tqdm(
        zip(df['participants_ID'].values,
            df['ec_eeg_path'].values,
            df['eo_eeg_path'].values),
        total=len(df)
    )
)

100%|██████████| 120/120 [00:01<00:00, 65.16it/s]


In [9]:
# save results in pickle file
bandpower_df = pd.DataFrame(res)
bandpower_df.to_pickle(f'../data/mdd_rtms_bandpower_all.pkl')
print("Shape of bandpower dataframe:", bandpower_df.shape)
print(bandpower_df.head())

Shape of bandpower dataframe: (120, 365)
  participants_ID  EC_Fp1_delta_power  EC_Fp1_theta_power  EC_Fp1_alpha_power  \
0    sub-87999321            0.440963            0.125691            0.169726   
1    sub-88000181            0.213105            0.201495            0.197796   
2    sub-88000313            0.198156            0.108387            0.435276   
3    sub-88000489            0.210357            0.230957            0.329855   
4    sub-88000533            0.299961            0.092930            0.089858   

   EC_Fp1_low_beta_power  EC_Fp1_high_beta_power  EC_Fp1_low_gamma_power  \
0               0.048599                0.053521                0.037638   
1               0.158821                0.061693                0.041036   
2               0.153407                0.047434                0.019371   
3               0.059994                0.055549                0.024703   
4               0.080159                0.118211                0.087667   

   EC_Fp1_high_

## DynaMix Weights

In [10]:
# load the pre-trained model
model = load_hf_model("dynamix-6d-alrnn-v1.0")

# set model to evaluation mode
model.eval()

# initialize the forecaster
forecaster = DynaMixForecaster(model)

In [11]:
def get_dynamix_latent(subj_data_path):

    channel_filter = ['F7', 'P8', 'T7', 'O2']
    # channel_filter = ['Fp1','Fz', 'F4', 'O1', 'Oz', 'O2',]

    eeg_dict = np.load(subj_data_path, allow_pickle=True)
    channel_labels = eeg_dict['labels']
    sampling_freq = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    assert eeg_data.shape[0] == len(channel_filter)

    offset = 5000
    CL = 10000
    T = 1000

    context_start = offset
    context_end = offset + CL

    # Load the time series data
    ts_data = eeg_data.T

    context_ts = ts_data[context_start:context_end,:] # context from 5000 to 15000 (10s-30s)

    # Convert to PyTorch tensor
    context_ts_tensor = torch.tensor(context_ts, dtype=torch.float32)
    
    activation = {}
    # a dict to store the activations
    def getActivation(name):
        activation[name] = list()
        # the hook signature
        def hook(model, input, output):
            activation[name].append(output.detach())
        return hook
    
    hooks = [
        forecaster.model.gating_network.register_forward_hook(getActivation("w_exp")),
        forecaster.model.gating_network.mlp_layer1.register_forward_hook(getActivation("mlp1")),
        forecaster.model.gating_network.mlp_layer2.register_forward_hook(getActivation("mlp2")),
        forecaster.model.register_forward_hook(getActivation("model_out")),
    ]

    # h = forecaster.model.gating_network.register_forward_hook(getActivation('w_exp'))

    # Make prediction
    with torch.no_grad():  # No gradient tracking needed for inference
        reconstruction_ts = forecaster.forecast(
            context=context_ts_tensor,
            horizon=T,
            preprocessing_method="pos_embedding",
            standardize=True,
            fit_nonstationary=False,
        )

    # h.remove()
    for h in hooks:
        h.remove()

    # w_exp = torch.stack(activation['w_exp']).numpy().squeeze()

    # return w_exp

    w_exp = torch.stack(activation["w_exp"]).numpy().squeeze()         # (T, 80)
    mlp1 = torch.stack(activation["mlp1"]).numpy().squeeze()           # (T, 80)
    mlp2 = torch.stack(activation["mlp2"]).numpy().squeeze()           # (T, 80)
    model_out = torch.stack(activation["model_out"]).numpy().squeeze() # (T, M)

    feats = np.concatenate([
        np.std(w_exp, axis=0),
        np.mean(w_exp, axis=0),
        np.std(mlp1, axis=0),
        np.mean(mlp1, axis=0),
        np.std(mlp2, axis=0),
        np.mean(mlp2, axis=0),
        np.std(model_out, axis=0),
        np.mean(model_out, axis=0),
    ])

    return feats

In [12]:
# extract features in parallel for all subjects
ec_res = Parallel(n_jobs=16)(delayed(get_dynamix_latent)(ec_path) for ec_path in tqdm(df['ec_eeg_path'].values))
eo_res = Parallel(n_jobs=16)(delayed(get_dynamix_latent)(ec_path) for ec_path in tqdm(df['eo_eeg_path'].values))

  0%|          | 0/120 [00:00<?, ?it/s]

 27%|██▋       | 32/120 [00:04<00:12,  7.11it/s]/users/scbao/.conda/envs/myenv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 120/120 [00:14<00:00,  8.07it/s]


In [13]:
print(np.array(ec_res).shape)
ec_res = np.array(ec_res)
eo_res = np.array(eo_res)
dynamix_latents = np.concatenate([eo_res, ec_res], axis=1)
print(dynamix_latents.shape)

(120, 500)
(120, 1000)


In [14]:
M = ec_res[0].shape[0] - 80*6
M = M // 2
print("M:", M)

col_names = (
    [f"dynamix_wexp_std_{i+1}" for i in range(80)] +
    [f"dynamix_wexp_mean_{i+1}" for i in range(80)] +
    [f"dynamix_mlp1_std_{i+1}" for i in range(80)] +
    [f"dynamix_mlp1_mean_{i+1}" for i in range(80)] +
    [f"dynamix_mlp2_std_{i+1}" for i in range(80)] +
    [f"dynamix_mlp2_mean_{i+1}" for i in range(80)] +
    [f"dynamix_modelout_std_{i+1}" for i in range(M)] +
    [f"dynamix_modelout_mean_{i+1}" for i in range(M)]
)

M: 10


In [15]:
df_dynamix = pd.DataFrame(np.vstack(ec_res), columns=col_names)
df_dynamix.insert(0, "participants_ID", df['participants_ID'].values)

df_dynamix.to_pickle('../data/dynamix_ec_all_no_stoch.pkl')
print(df_dynamix.shape)
print(df_dynamix.head())

(120, 501)
  participants_ID  dynamix_wexp_std_1  dynamix_wexp_std_2  dynamix_wexp_std_3  \
0    sub-87999321        3.276108e-07            0.005008            0.025298   
1    sub-88000181        5.328037e-07            0.005720            0.023091   
2    sub-88000313        2.354025e-07            0.002576            0.008679   
3    sub-88000489        2.926931e-06            0.004610            0.019825   
4    sub-88000533        1.299273e-06            0.006225            0.027154   

   dynamix_wexp_std_4  dynamix_wexp_std_5  dynamix_wexp_std_6  \
0            0.010002            0.003433        4.064104e-07   
1            0.012928            0.004187        5.787883e-07   
2            0.005157            0.001821        2.820382e-07   
3            0.005195            0.002675        3.318399e-06   
4            0.024236            0.008621        1.446256e-06   

   dynamix_wexp_std_7  dynamix_wexp_std_8  dynamix_wexp_std_9  ...  \
0            0.009988            0.000220

In [16]:
# get ec, std result and save in pickle file
# ec_std = np.std(ec_res, axis=1)

# col_names = [f"dynamix_ec_std_{i+1}" for i in range(80)]
# df_dynamix = pd.DataFrame(ec_std, columns=col_names)
# df_dynamix.insert(0, "participants_ID", df['participants_ID'].values)

# df_dynamix.to_pickle(f'../data/dynamix_ec_wexp_all.pkl')
# print("Shape of dynamix dataframe:", df_dynamix.shape)
# print(df_dynamix.head())